#Environment Setup

In [1]:
!pip install fastapi uvicorn python-multipart scikit-learn joblib pytesseract nest-asyncio pyngrok
!sudo apt install tesseract-ocr

import os
import joblib
from PIL import Image
import pytesseract

os.makedirs('training_data/invoices', exist_ok=True)
os.makedirs('training_data/receipts', exist_ok=True)
os.makedirs('training_data/contracts', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('api', exist_ok=True)

print("Environment and folders ready.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Environment and folders ready.


In [6]:
from PIL import Image, ImageDraw, ImageFont
import os

def create_synthetic_doc(text, path):
    img = Image.new('RGB', (500, 700), color='white')
    d = ImageDraw.Draw(img)
    d.text((50, 50), text, fill=(0, 0, 0))
    img.save(path)

data_map = {
    "invoices": ["INVOICE", "Bill To:", "Due Date", "Total Amount Due", "Invoice Number", "Tax ID"],
    "receipts": ["RECEIPT", "Items Purchased", "Cashier:", "Change Due", "Thank you for your visit", "Subtotal"],
    "contracts": ["CONTRACT AGREEMENT", "Terms and Conditions", "Signature", "Witness", "Hereby agreed", "Parties"]
}

for category, keywords in data_map.items():
    for i in range(15):
        content = f"{category.upper()} #{i}\n" + " ".join(keywords * 3)
        file_path = f'training_data/{category}/{category}_{i}.jpg'
        create_synthetic_doc(content, file_path)

print("Synthetic dataset created: 45 images ready for OCR.")

Synthetic dataset created: 45 images ready for OCR.


# Document Classification

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

def load_documents(data_dir):
    documents = []
    labels = []
    for doc_type in os.listdir(data_dir):
        folder_path = os.path.join(data_dir, doc_type)
        if not os.path.isdir(folder_path):
            continue
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                img = Image.open(file_path)
                text = pytesseract.image_to_string(img)
                documents.append(text)
                labels.append(doc_type)
            except Exception as e:
                print(f"Error processing {filename}: {e}")
    return documents, labels

documents, labels = load_documents('training_data')
print(f'Loaded {len(documents)} documents')
print(f'Classes: {set(labels)}')

Loaded 45 documents
Classes: {'contracts', 'invoices', 'receipts'}


## 1.2 Train and Save the Classifie

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42, stratify=labels
)

vectorizer = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_vec, y_train)

y_pred = classifier.predict(X_test_vec)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2%}')
print(classification_report(y_test, y_pred))

joblib.dump(vectorizer, 'models/vectorizer.pkl')
joblib.dump(classifier, 'models/classifier.pkl')
print("Models saved successfully!")

Accuracy: 100.00%
              precision    recall  f1-score   support

   contracts       1.00      1.00      1.00         3
    invoices       1.00      1.00      1.00         3
    receipts       1.00      1.00      1.00         3

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9

Models saved successfully!


# Build the REST API

In [19]:
%%writefile api/main.py
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from PIL import Image
import pytesseract
import joblib
import io
import os

app = FastAPI(title='Document Intelligence API', version='1.0.0')

# Load models from the absolute path to avoid confusion
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
vectorizer_path = os.path.join(BASE_DIR, 'models', 'vectorizer.pkl')
classifier_path = os.path.join(BASE_DIR, 'models', 'classifier.pkl')

vectorizer = joblib.load(vectorizer_path)
classifier = joblib.load(classifier_path)

@app.get('/')
def root():
    return {'message': 'Document Intelligence API', 'version': '1.0.0'}

@app.post('/classify')
async def classify_document(file: UploadFile = File(...)):
    try:
        contents = await file.read()
        image = Image.open(io.BytesIO(contents))

        # Extract text via OCR
        text = pytesseract.image_to_string(image)

        if not text.strip():
            return {"error": "No text could be extracted from the image."}

        # Predict
        text_vec = vectorizer.transform([text])
        prediction = classifier.predict(text_vec)[0]
        probabilities = classifier.predict_proba(text_vec)[0]
        confidence = max(probabilities)

        return {
            'document_type': str(prediction),
            'confidence': float(confidence),
            'text_snippet': text[:100] # Returns first 100 chars for verification
        }
    except Exception as e:
        return JSONResponse(status_code=500, content={'error': str(e)})

Overwriting api/main.py


# Deploy and Test

In [21]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import asyncio
nest_asyncio.apply()
ngrok.kill()
# i have removed the authentication token
NGROK_AUTH_TOKEN = ""
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000).public_url
print(f" Public URL: {public_url}")
print(f" Interactive Swagger Docs: {public_url}/docs")

config = uvicorn.Config("api.main:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

await server.serve()

 Public URL: https://spout-pacifier-splatter.ngrok-free.dev
 Interactive Swagger Docs: https://spout-pacifier-splatter.ngrok-free.dev/docs


INFO:     Started server process [4035]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [4035]
